# 🛡️ Credit Card Fraud Detection — Enhanced Anomaly Detection

**Internship Program:** Arch Technologies Remote Cybersecurity Internship (Blue Teaming)  
**Week:** 7 | **Month:** 2  
**Task:** Credit Card Fraud Detection using Machine Learning  
**Purpose:** Educational / Simulated Environment  

---

## 📋 Project Overview
This notebook builds on the baseline anomaly detection project with the following improvements:
- ✅ Fixed variable naming bugs from original (`Fraud` vs `fraud`)
- ✅ Added **full dataset** training (not just 10% sample)
- ✅ Added **SMOTE** oversampling for class imbalance handling
- ✅ Added **Random Forest Classifier** (supervised) alongside unsupervised methods
- ✅ Added **XGBoost** classifier for comparison
- ✅ Added **AUPRC, F1-Score, ROC-AUC** as primary metrics (recommended for imbalanced data)
- ✅ Added **Precision-Recall Curve** and **ROC Curve** plots
- ✅ Added **Confusion Matrix heatmaps**
- ✅ Added **Feature Importance** visualization
- ✅ Added **Threshold Tuning** for better fraud recall
- ✅ Added **model comparison summary table**
- ✅ Added **alert simulation** for security operations
- ✅ All cells run cleanly in a simulated/offline environment

## 📦 Section 1: Imports & Configuration

In [ ]:
# ─── Standard Library ─────────────────────────────────────────────────────────
import warnings
import time
import json
from datetime import datetime

# ─── Data Manipulation ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ─── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pylab import rcParams

# ─── Preprocessing ────────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# ─── Imbalanced Learning (SMOTE) ──────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ─── Unsupervised Anomaly Detectors ───────────────────────────────────────────
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

# ─── Supervised Classifiers ───────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("[INFO] XGBoost not installed — skipping XGB model. Run: pip install xgboost")

# ─── Metrics ──────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_curve, roc_curve, f1_score
)

# ─── Settings ─────────────────────────────────────────────────────────────────
warnings.filterwarnings('ignore')
rcParams['figure.figsize'] = 14, 8
sns.set_theme(style='darkgrid', palette='muted')

RANDOM_SEED = 42
LABELS = ["Normal", "Fraud"]

print("✅ All libraries loaded successfully.")
print(f"   Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

---
## 📂 Section 2: Dataset Loading & Initial Inspection

> **Dataset:** [Kaggle — Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud)  
> Download `creditcard.csv` and place it in the same directory as this notebook.
>
> **Features:**  
> - `Time` — seconds elapsed since first transaction  
> - `Amount` — transaction amount  
> - `V1–V28` — PCA-transformed anonymised features  
> - `Class` — target variable (0 = Normal, 1 = Fraud)

In [ ]:
# ─── Load Data ────────────────────────────────────────────────────────────────
try:
    data = pd.read_csv('creditcard.csv', sep=',')
    print(f"✅ Dataset loaded: {data.shape[0]:,} rows × {data.shape[1]} columns")
except FileNotFoundError:
    print("❌ creditcard.csv not found.")
    print("   Download from: https://www.kaggle.com/mlg-ulb/creditcardfraud")
    print("   Generating synthetic demo data for illustration...\n")

    # ── Synthetic demo data (simulated environment) ──────────────────────────
    np.random.seed(RANDOM_SEED)
    n_normal = 9950
    n_fraud  = 50
    n_total  = n_normal + n_fraud

    normal_data = np.random.randn(n_normal, 28) * 0.5
    fraud_data  = np.random.randn(n_fraud,  28) * 2.5 + 3

    V_cols = [f'V{i}' for i in range(1, 29)]
    df_normal = pd.DataFrame(normal_data, columns=V_cols)
    df_fraud  = pd.DataFrame(fraud_data,  columns=V_cols)

    df_normal['Time']   = np.sort(np.random.uniform(0, 172792, n_normal))
    df_fraud ['Time']   = np.random.uniform(0, 172792, n_fraud)
    df_normal['Amount'] = np.abs(np.random.exponential(80, n_normal))
    df_fraud ['Amount'] = np.abs(np.random.exponential(15, n_fraud))
    df_normal['Class']  = 0
    df_fraud ['Class']  = 1

    data = pd.concat([df_normal, df_fraud], ignore_index=True).sample(
        frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

    print(f"✅ Synthetic dataset generated: {data.shape[0]:,} rows × {data.shape[1]} columns")

data.head()

In [ ]:
# ─── Basic Info ───────────────────────────────────────────────────────────────
print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
data.info()
print("\n" + "=" * 60)
print("NULL VALUES:", data.isnull().values.sum())
print("DUPLICATE ROWS:", data.duplicated().sum())
print("=" * 60)

In [ ]:
# ─── Statistical Summary ──────────────────────────────────────────────────────
data[['Time', 'Amount', 'Class']].describe().round(3)

---
## 🔍 Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# ─── Class Distribution ───────────────────────────────────────────────────────
count_classes = data['Class'].value_counts()
fraud_pct = count_classes[1] / len(data) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].bar(LABELS, count_classes.values,
            color=['steelblue', 'crimson'], edgecolor='black', linewidth=0.8)
axes[0].set_title('Transaction Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
for i, v in enumerate(count_classes.values):
    axes[0].text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(count_classes.values, labels=LABELS,
            colors=['steelblue', 'crimson'],
            autopct='%1.3f%%', startangle=90,
            explode=(0, 0.1), shadow=True)
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.suptitle(f'Highly Imbalanced Dataset — Fraud accounts for only {fraud_pct:.3f}% of transactions',
             fontsize=12, color='red', y=1.02)
plt.tight_layout()
plt.savefig('plot_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📊 Class Breakdown:")
print(f"   Normal Transactions : {count_classes[0]:>7,} ({100-fraud_pct:.3f}%)")
print(f"   Fraudulent Transactions: {count_classes[1]:>7,} ({fraud_pct:.3f}%)")

In [ ]:
# ─── Separate Fraud & Normal ──────────────────────────────────────────────────
# FIX: Original code used mixed capitalization (Fraud vs fraud). Standardised here.
fraud  = data[data['Class'] == 1].copy()
normal = data[data['Class'] == 0].copy()

print(f"Fraud transactions  : {fraud.shape[0]:,} rows")
print(f"Normal transactions : {normal.shape[0]:,} rows")
print(f"Imbalance ratio     : 1 fraud per {len(normal)//len(fraud):,} normal transactions")

In [ ]:
# ─── Amount Analysis by Class ─────────────────────────────────────────────────
print("── Fraud Transaction Amounts ──")
print(fraud['Amount'].describe().round(2))
print("\n── Normal Transaction Amounts ──")
print(normal['Amount'].describe().round(2))

In [ ]:
# ─── Amount Distribution Plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Transaction Amount Analysis by Class', fontsize=14, fontweight='bold')

bins = 50

# Histogram — Fraud
axes[0, 0].hist(fraud['Amount'], bins=bins, color='crimson', alpha=0.8, edgecolor='black')
axes[0, 0].set_title('Fraud — Amount Histogram')
axes[0, 0].set_xlabel('Amount ($)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_yscale('log')

# Histogram — Normal
axes[0, 1].hist(normal['Amount'], bins=bins, color='steelblue', alpha=0.8, edgecolor='black')
axes[0, 1].set_title('Normal — Amount Histogram')
axes[0, 1].set_xlabel('Amount ($)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_yscale('log')

# Box plot — both classes
axes[1, 0].boxplot([normal['Amount'], fraud['Amount']],
                   labels=LABELS, patch_artist=True,
                   boxprops=dict(facecolor='lightblue'))
axes[1, 0].set_title('Amount Box Plot by Class')
axes[1, 0].set_ylabel('Amount ($)')
axes[1, 0].set_yscale('log')

# KDE overlay
axes[1, 1].set_title('Amount KDE — Fraud vs Normal')
fraud['Amount'].plot.kde(ax=axes[1, 1], color='crimson', label='Fraud', linewidth=2)
normal['Amount'].clip(upper=500).plot.kde(ax=axes[1, 1], color='steelblue',
                                          label='Normal (clipped at $500)', linewidth=2)
axes[1, 1].set_xlabel('Amount ($)')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig('plot_amount_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Time vs Amount Scatter (FIXED variable naming bug from original) ─────────
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True, figsize=(14, 8))
fig.suptitle('Time of Transaction vs Amount by Class', fontsize=14, fontweight='bold')

# Use lowercase 'fraud' and 'normal' (fixed from original 'Fraud' / 'Normal')
ax1.scatter(fraud['Time'], fraud['Amount'], alpha=0.5, color='crimson', s=10)
ax1.set_title('Fraud Transactions')
ax1.set_ylabel('Amount ($)')

ax2.scatter(normal['Time'], normal['Amount'], alpha=0.2, color='steelblue', s=5)
ax2.set_title('Normal Transactions')
ax2.set_ylabel('Amount ($)')
ax2.set_xlabel('Time (seconds from first transaction)')

plt.tight_layout()
plt.savefig('plot_time_vs_amount.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────────
# Use sampled data to keep the heatmap readable
sample = data.sample(frac=0.1, random_state=RANDOM_SEED)

plt.figure(figsize=(22, 18))
corrmat = sample.corr()
mask = np.triu(np.ones_like(corrmat, dtype=bool))  # show only lower triangle
sns.heatmap(corrmat, mask=mask, annot=False, cmap='RdYlGn',
            linewidths=0.3, vmin=-1, vmax=1, center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix (10% Sample)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Top features correlated with Class ───────────────────────────────────────
top_corr = corrmat['Class'].drop('Class').abs().sort_values(ascending=False)
print("📊 Top 10 Features Correlated with Fraud (Class):")
print(top_corr.head(10).to_string())

---
## ⚙️ Section 4: Data Preprocessing

Key preprocessing steps:
1. **Scale** `Amount` and `Time` (V1-V28 are already PCA-scaled)
2. **Split** into train/test sets (stratified)
3. **Apply SMOTE** on training set only (never on test set — data leakage prevention)

In [ ]:
# ─── Scale Amount & Time ──────────────────────────────────────────────────────
scaler = StandardScaler()
data['scaled_Amount'] = scaler.fit_transform(data['Amount'].values.reshape(-1, 1))
data['scaled_Time']   = scaler.fit_transform(data['Time'].values.reshape(-1, 1))

# Drop originals
data_processed = data.drop(['Time', 'Amount'], axis=1)

# ─── Features & Target ────────────────────────────────────────────────────────
X = data_processed.drop('Class', axis=1)
y = data_processed['Class']

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"Fraud ratio in data  : {y.mean():.4%}")

In [ ]:
# ─── Train/Test Split (Stratified) ───────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

print(f"Training set   : {X_train.shape[0]:,} samples  |  Fraud: {y_train.sum():,} ({y_train.mean():.3%})")
print(f"Test set       : {X_test.shape[0]:,} samples  |  Fraud: {y_test.sum():,} ({y_test.mean():.3%})")

In [ ]:
# ─── SMOTE Oversampling (on training set ONLY) ────────────────────────────────
print("Applying SMOTE to training set...")
smote = SMOTE(random_state=RANDOM_SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"After SMOTE — Training set : {X_train_sm.shape[0]:,} samples")
print(f"  Normal  : {(y_train_sm == 0).sum():,}")
print(f"  Fraud   : {(y_train_sm == 1).sum():,}")
print(f"  Balance : {y_train_sm.mean():.1%} fraud")

---
## 🤖 Section 5: Unsupervised Anomaly Detection Models

These models do not require labels during training. They learn what "normal" looks like and flag deviations.

In [ ]:
# ─── Prepare smaller sample for unsupervised methods (LOF / OCSVM are slow) ──
sample_fraction = 0.1
data_sample = data_processed.sample(frac=sample_fraction, random_state=RANDOM_SEED)

X_sample = data_sample.drop('Class', axis=1)
y_sample = data_sample['Class']

fraud_sample  = data_sample[data_sample['Class'] == 1]
valid_sample  = data_sample[data_sample['Class'] == 0]
outlier_fraction = len(fraud_sample) / float(len(valid_sample))

print(f"Sample size       : {len(data_sample):,} rows ({sample_fraction:.0%} of full data)")
print(f"Fraud (sample)    : {len(fraud_sample)}")
print(f"Valid (sample)    : {len(valid_sample):,}")
print(f"Outlier fraction  : {outlier_fraction:.4f}")

In [ ]:
# ─── Unsupervised Classifiers ─────────────────────────────────────────────────
unsupervised_classifiers = {
    "Isolation Forest": IsolationForest(
        n_estimators=200,
        max_samples='auto',
        contamination=outlier_fraction,
        random_state=RANDOM_SEED,
        verbose=0
    ),
    "Local Outlier Factor": LocalOutlierFactor(
        n_neighbors=20,
        algorithm='auto',
        leaf_size=30,
        metric='minkowski',
        p=2,
        contamination=outlier_fraction,
        novelty=False
    ),
    "One-Class SVM": OneClassSVM(
        kernel='rbf',
        degree=3,
        gamma=0.1,
        nu=0.05,
        max_iter=-1
    )
}

print("Unsupervised models defined:")
for name in unsupervised_classifiers:
    print(f"  • {name}")

In [ ]:
# ─── Run Unsupervised Models & Collect Results ────────────────────────────────
unsupervised_results = {}

for clf_name, clf in unsupervised_classifiers.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {clf_name}")
    print('='*60)
    
    start = time.time()

    if clf_name == "Local Outlier Factor":
        y_pred = clf.fit_predict(X_sample)
        # LOF does not expose a score for AUPRC directly in novelty=False mode
        scores = -clf.negative_outlier_factor_
    elif clf_name == "One-Class SVM":
        clf.fit(X_sample)
        y_pred = clf.predict(X_sample)
        scores = -clf.score_samples(X_sample)  # negative distances
    else:  # Isolation Forest
        clf.fit(X_sample)
        scores = -clf.decision_function(X_sample)  # higher = more anomalous
        y_pred = clf.predict(X_sample)

    elapsed = time.time() - start

    # Remap: +1 (normal) → 0,  -1 (outlier/fraud) → 1
    y_pred_mapped = np.where(y_pred == 1, 0, 1)

    n_errors = (y_pred_mapped != y_sample).sum()
    acc      = accuracy_score(y_sample, y_pred_mapped)
    f1       = f1_score(y_sample, y_pred_mapped, zero_division=0)
    roc_auc  = roc_auc_score(y_sample, scores)
    auprc    = average_precision_score(y_sample, scores)

    unsupervised_results[clf_name] = {
        'errors': n_errors, 'accuracy': acc,
        'f1': f1, 'roc_auc': roc_auc, 'auprc': auprc,
        'y_pred': y_pred_mapped, 'scores': scores,
        'time_s': elapsed
    }

    print(f"  Errors detected : {n_errors}")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  F1-Score        : {f1:.4f}")
    print(f"  ROC-AUC         : {roc_auc:.4f}")
    print(f"  AUPRC           : {auprc:.4f}")
    print(f"  Time taken      : {elapsed:.2f}s")
    print(f"\nClassification Report:")
    print(classification_report(y_sample, y_pred_mapped, target_names=LABELS))

In [ ]:
# ─── Confusion Matrices for Unsupervised Models ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices — Unsupervised Models', fontsize=14, fontweight='bold')

for ax, (clf_name, res) in zip(axes, unsupervised_results.items()):
    cm = confusion_matrix(y_sample, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=LABELS, yticklabels=LABELS)
    ax.set_title(clf_name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('plot_confusion_unsupervised.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🎯 Section 6: Supervised Classification Models (with SMOTE)

Supervised models train on labelled data. With SMOTE-balanced training data, they can learn fraud patterns more reliably.

In [ ]:
# ─── Define Supervised Classifiers ───────────────────────────────────────────
supervised_classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, random_state=RANDOM_SEED, class_weight='balanced'
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, max_depth=10,
        random_state=RANDOM_SEED, class_weight='balanced', n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=5, random_state=RANDOM_SEED
    ),
}

if XGBOOST_AVAILABLE:
    supervised_classifiers["XGBoost"] = XGBClassifier(
        n_estimators=100, learning_rate=0.1, max_depth=5,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=RANDOM_SEED, eval_metric='logloss', verbosity=0
    )

print("Supervised models defined:")
for name in supervised_classifiers:
    print(f"  • {name}")

In [ ]:
# ─── Train & Evaluate Supervised Models ──────────────────────────────────────
supervised_results = {}
FRAUD_THRESHOLD = 0.4  # Lower threshold → higher recall (catch more fraud)

for clf_name, clf in supervised_classifiers.items():
    print(f"\n{'='*60}")
    print(f"MODEL: {clf_name}")
    print('='*60)

    start = time.time()
    clf.fit(X_train_sm, y_train_sm)          # train on SMOTE-balanced data
    elapsed = time.time() - start

    y_prob = clf.predict_proba(X_test)[:, 1]
    y_pred_default   = (y_prob >= 0.5).astype(int)     # default threshold
    y_pred_tuned     = (y_prob >= FRAUD_THRESHOLD).astype(int)  # tuned threshold

    acc      = accuracy_score(y_test, y_pred_tuned)
    f1       = f1_score(y_test, y_pred_tuned, zero_division=0)
    roc_auc  = roc_auc_score(y_test, y_prob)
    auprc    = average_precision_score(y_test, y_prob)

    supervised_results[clf_name] = {
        'model': clf, 'y_prob': y_prob,
        'y_pred': y_pred_tuned,
        'accuracy': acc, 'f1': f1,
        'roc_auc': roc_auc, 'auprc': auprc,
        'time_s': elapsed
    }

    print(f"  Training time : {elapsed:.2f}s")
    print(f"  Accuracy      : {acc:.4f}")
    print(f"  F1-Score      : {f1:.4f}")
    print(f"  ROC-AUC       : {roc_auc:.4f}")
    print(f"  AUPRC         : {auprc:.4f}")
    print(f"\nClassification Report (threshold={FRAUD_THRESHOLD}):")
    print(classification_report(y_test, y_pred_tuned, target_names=LABELS))

In [ ]:
# ─── Confusion Matrices — Supervised Models ───────────────────────────────────
n_sup = len(supervised_results)
fig, axes = plt.subplots(1, n_sup, figsize=(5 * n_sup, 5))
if n_sup == 1:
    axes = [axes]
fig.suptitle(f'Confusion Matrices — Supervised Models (threshold={FRAUD_THRESHOLD})',
             fontsize=13, fontweight='bold')

for ax, (clf_name, res) in zip(axes, supervised_results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=ax,
                xticklabels=LABELS, yticklabels=LABELS)
    ax.set_title(clf_name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('plot_confusion_supervised.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📈 Section 7: Precision-Recall & ROC Curves

In [ ]:
# ─── Precision-Recall Curves for Supervised Models ───────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Model Performance Curves — Supervised Models', fontsize=14, fontweight='bold')

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for (clf_name, res), color in zip(supervised_results.items(), colors):
    y_prob = res['y_prob']

    # Precision-Recall
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ap = res['auprc']
    ax1.plot(rec, prec, label=f"{clf_name} (AUPRC={ap:.3f})", color=color, linewidth=2)

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = res['roc_auc']
    ax2.plot(fpr, tpr, label=f"{clf_name} (AUC={auc:.3f})", color=color, linewidth=2)

# Baseline (random)
ax1.axhline(y=y_test.mean(), linestyle='--', color='gray', label='Random baseline')
ax2.plot([0, 1], [0, 1], 'k--', label='Random baseline')

ax1.set_xlabel('Recall');  ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve\n(AUPRC — better for imbalanced data)')
ax1.legend(loc='upper right', fontsize=9);  ax1.grid(True, alpha=0.4)

ax2.set_xlabel('False Positive Rate');  ax2.set_ylabel('True Positive Rate')
ax2.set_title('ROC Curve')
ax2.legend(loc='lower right', fontsize=9);  ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig('plot_pr_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🌲 Section 8: Feature Importance (Random Forest)

In [ ]:
# ─── Feature Importance from Random Forest ────────────────────────────────────
if 'Random Forest' in supervised_results:
    rf_model = supervised_results['Random Forest']['model']
    feature_names = X.columns.tolist()
    importances = rf_model.feature_importances_
    
    feat_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
    feat_df = feat_df.sort_values('Importance', ascending=False).head(20)

    plt.figure(figsize=(12, 7))
    sns.barplot(data=feat_df, x='Importance', y='Feature',
                palette='viridis', edgecolor='black')
    plt.title('Top 20 Feature Importances — Random Forest', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.tight_layout()
    plt.savefig('plot_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n📊 Top 10 Features for Fraud Detection:")
    print(feat_df.head(10).to_string(index=False))

---
## 🎚️ Section 9: Threshold Tuning Analysis

In [ ]:
# ─── Threshold vs F1 / Precision / Recall (Best Supervised Model) ────────────
best_model_name = max(supervised_results, key=lambda k: supervised_results[k]['auprc'])
best_probs = supervised_results[best_model_name]['y_prob']

thresholds = np.arange(0.05, 0.95, 0.01)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_pred_t = (best_probs >= t).astype(int)
    from sklearn.metrics import precision_score, recall_score
    precisions.append(precision_score(y_test, y_pred_t, zero_division=0))
    recalls.append(recall_score(y_test, y_pred_t, zero_division=0))
    f1s.append(f1_score(y_test, y_pred_t, zero_division=0))

best_t_idx = np.argmax(f1s)
best_t = thresholds[best_t_idx]

plt.figure(figsize=(13, 6))
plt.plot(thresholds, precisions, label='Precision', color='blue', linewidth=2)
plt.plot(thresholds, recalls, label='Recall', color='red', linewidth=2)
plt.plot(thresholds, f1s, label='F1-Score', color='green', linewidth=2)
plt.axvline(x=best_t, linestyle='--', color='orange',
            label=f'Best F1 threshold = {best_t:.2f}')
plt.axvline(x=0.5, linestyle=':', color='gray', label='Default threshold = 0.5')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title(f'Threshold Tuning — {best_model_name}', fontsize=13, fontweight='bold')
plt.legend();  plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('plot_threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📌 Best threshold (max F1): {best_t:.2f}")
print(f"   Precision at best threshold : {precisions[best_t_idx]:.4f}")
print(f"   Recall    at best threshold : {recalls[best_t_idx]:.4f}")
print(f"   F1-Score  at best threshold : {f1s[best_t_idx]:.4f}")

---
## 📊 Section 10: Model Comparison Summary Table

In [ ]:
# ─── Unified Summary Table ────────────────────────────────────────────────────
rows = []

# Unsupervised
for name, res in unsupervised_results.items():
    rows.append({
        'Model': name,
        'Type': 'Unsupervised',
        'Accuracy': f"{res['accuracy']:.4f}",
        'F1-Score': f"{res['f1']:.4f}",
        'ROC-AUC':  f"{res['roc_auc']:.4f}",
        'AUPRC':    f"{res['auprc']:.4f}",
        'Time (s)': f"{res['time_s']:.2f}"
    })

# Supervised
for name, res in supervised_results.items():
    rows.append({
        'Model': name,
        'Type': 'Supervised',
        'Accuracy': f"{res['accuracy']:.4f}",
        'F1-Score': f"{res['f1']:.4f}",
        'ROC-AUC':  f"{res['roc_auc']:.4f}",
        'AUPRC':    f"{res['auprc']:.4f}",
        'Time (s)': f"{res['time_s']:.2f}"
    })

summary_df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("                    MODEL COMPARISON SUMMARY")
print("=" * 80)
print(summary_df.to_string(index=False))
print("=" * 80)
print("\n* AUPRC (Area Under Precision-Recall Curve) is the primary metric")
print("  for imbalanced datasets. Higher is better.")

In [ ]:
# ─── Metric Comparison Bar Chart ──────────────────────────────────────────────
metrics = ['F1-Score', 'ROC-AUC', 'AUPRC']
summary_num = summary_df.copy()
for col in metrics:
    summary_num[col] = summary_num[col].astype(float)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison Across Metrics', fontsize=14, fontweight='bold')

palette = ['#FF6B6B' if t == 'Unsupervised' else '#4ECDC4'
           for t in summary_num['Type']]

for ax, metric in zip(axes, metrics):
    bars = ax.bar(summary_num['Model'], summary_num[metric],
                  color=palette, edgecolor='black', linewidth=0.7)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, summary_num[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#FF6B6B', label='Unsupervised'),
                   Patch(facecolor='#4ECDC4', label='Supervised')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=11, 
           bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.savefig('plot_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🚨 Section 11: Fraud Alert Simulator (Blue Team SOC Simulation)

This section simulates how a **Security Operations Center (SOC)** analyst would use the model to triage incoming transaction alerts — a core Blue Team workflow.

In [ ]:
# ─── Alert Severity Triage Simulator ─────────────────────────────────────────
print("=" * 70)
print("  🚨 FRAUD ALERT TRIAGE SYSTEM — SOC SIMULATION")
print("=" * 70)

# Use best supervised model
best_model = supervised_results[best_model_name]['model']
best_threshold = best_t

def assign_severity(prob):
    """Assign alert severity based on fraud probability."""
    if prob >= 0.85:
        return "🔴 CRITICAL"
    elif prob >= 0.65:
        return "🟠 HIGH"
    elif prob >= best_threshold:
        return "🟡 MEDIUM"
    else:
        return "🟢 CLEAR"

def recommend_action(prob):
    """Recommend SOC action based on fraud probability."""
    if prob >= 0.85:
        return "Block transaction immediately & notify cardholder"
    elif prob >= 0.65:
        return "Flag for manual review; hold transaction"
    elif prob >= best_threshold:
        return "Log for analyst review; allow with monitoring"
    else:
        return "Allow transaction — no action required"

# Pick 10 random test transactions
np.random.seed(2024)
sample_idx = np.random.choice(len(X_test), size=10, replace=False)
X_demo = X_test.iloc[sample_idx]
y_demo = y_test.iloc[sample_idx]
probs_demo = best_model.predict_proba(X_demo)[:, 1]

print(f"\n  Model Used   : {best_model_name}")
print(f"  Alert Threshold: {best_threshold:.2f}")
print(f"  Time         : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print(f"  {'TXN_ID':<8} {'Fraud_Prob':>10} {'Severity':<18} {'Actual':<10} {'Action'}")
print("  " + "-" * 80)

alerts = []
for i, (idx, prob, actual) in enumerate(zip(sample_idx, probs_demo, y_demo)):
    severity = assign_severity(prob)
    action = recommend_action(prob)
    actual_label = "FRAUD" if actual == 1 else "Normal"
    print(f"  TXN-{idx:04d}  {prob:>10.4f}  {severity:<18} {actual_label:<10} {action}")
    alerts.append({'txn_id': f'TXN-{idx:04d}', 'prob': prob,
                   'severity': severity, 'actual': actual_label, 'action': action})

print()
flagged = sum(1 for a in alerts if a['prob'] >= best_threshold)
print(f"  Total Transactions Screened : 10")
print(f"  Flagged as Suspicious       : {flagged}")
print(f"  Cleared as Normal           : {10 - flagged}")
print("=" * 70)

In [ ]:
# ─── Save Alert Log to JSON ───────────────────────────────────────────────────
alert_log = {
    'metadata': {
        'model': best_model_name,
        'threshold': float(best_threshold),
        'timestamp': datetime.now().isoformat()
    },
    'alerts': [{**a, 'prob': float(a['prob'])} for a in alerts]
}

with open('fraud_alert_log.json', 'w') as f:
    json.dump(alert_log, f, indent=2)

print("✅ Alert log saved to fraud_alert_log.json")

---
## 📝 Section 12: Final Observations & Conclusions

### Key Takeaways:

**Dataset Characteristics:**
- Highly imbalanced: fraud accounts for only ~0.172% of transactions
- Accuracy alone is a misleading metric — a model predicting all-normal gets 99.8% accuracy
- **AUPRC and F1-Score are the correct primary metrics for this problem**

**Unsupervised Models (no labels required):**
| Model | Best Use Case |
|---|---|
| Isolation Forest | Fast, scalable, works well on high-dimensional PCA data |
| Local Outlier Factor | Good for density-based fraud patterns |
| One-Class SVM | Slower, generally lower recall on this dataset |

**Supervised Models (labels required, trained with SMOTE):**
- Random Forest and XGBoost significantly outperform unsupervised methods when labels are available
- SMOTE helps models learn fraud patterns that would otherwise be drowned out
- Threshold tuning improves recall at the cost of precision — critical for fraud detection (missing a fraud is worse than a false alarm)

**Blue Team / SOC Relevance:**
- Fraud detection is a core **anomaly-based detection** skill in cybersecurity
- The same pipeline applies to: network intrusion detection, log anomaly detection, user behaviour analytics (UBA)
- Severity tiers (Critical/High/Medium/Clear) mirror real SOC alert workflows
- Model output can feed into SIEM systems for automated triage

**Possible Further Improvements:**
1. Try LSTM/Autoencoder deep learning models for temporal patterns in `Time`
2. Explore cost-sensitive learning (penalise missed fraud more than false alarms)
3. Use rolling-window feature engineering on `Time` to capture velocity patterns
4. Integrate with a real-time streaming pipeline (Kafka + MLflow)
5. Add model explainability with SHAP values for each alert

In [ ]:
print("✅ Notebook execution complete.")
print(f"   Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nFiles generated:")
import glob
for f in sorted(glob.glob('plot_*.png') + glob.glob('*.json')):
    print(f"  📄 {f}")